# Personal Desktop Computer Electricity Consumption
## A Full-Year Exploratory Data Analysis of Computing Behavior

**Author:** Uddipan Dasgupta — PhD Candidate, Amity University Kolkata  
**Data collection period:** 13 March 2025 – 13 March 2026

---

> **Reproducible companion notebook** for the research paper of the same title.  
> Run all cells top-to-bottom to regenerate every statistic and figure.  

### Project directory layout
```
project/
├── data/
│   ├── final_combined_power_data.csv   ← raw daily kWh readings
│   └── univ_dates.txt                  ← university visit dates (one per line)
├── figures/                            ← all output figures (auto-created)
└── pc_energy_analysis_notebook.ipynb
```


---
## 1 · Environment Setup & Imports

All third-party libraries required by the notebook. Standard library only;  
no additional `pip install` steps are needed beyond a typical scientific Python stack  
(`numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`, `seaborn`).


In [127]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
from scipy.stats      import gaussian_kde
from sklearn.cluster  import KMeans

import matplotlib
import matplotlib.pyplot   as plt
import matplotlib.patches  as mpatches
import matplotlib.dates    as mdates
import seaborn             as sns

# ── Output directories (created if absent) ────────────────────────────────────
os.makedirs("figures", exist_ok=True)
os.makedirs("data",    exist_ok=True)

print("Environment ready.")
print(f"  pandas  {pd.__version__}  |  numpy {np.__version__}  |  matplotlib {matplotlib.__version__}")


Environment ready.
  pandas  2.3.1  |  numpy 2.2.6  |  matplotlib 3.10.3


---
## 2 · Global Plotting Configuration

All aesthetic parameters, the shared colour palette, and the two helper functions
(`shade_travel`, `savefig`) are defined here so that every subsequent figure cell
inherits consistent styling automatically.

### Figure export format
Set `FIG_FORMAT = "svg"` to export vector graphics instead of raster PNG.  
All `savefig()` calls resolve the extension from this single variable.


In [128]:
# ── Figure export format ──────────────────────────────────────────────────────
# Change this one variable to switch the output format for every figure.
FIG_FORMAT = "svg"   # "png" | "svg" | "pdf"

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F8F9FA",
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "grid.color":       "#CCCCCC",
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "figure.dpi":      300,
})

# ── Shared colour palette ─────────────────────────────────────────────────────
HOME_COLOR   = "#2196F3"   # blue   — Home Days
UNIV_COLOR   = "#FF5722"   # orange — University Days
IDLE_COLOR   = "#9C27B0"   # purple — Idle state
MOD_COLOR    = "#4CAF50"   # green  — Moderate state
HEAVY_COLOR  = "#F44336"   # red    — Heavy state
TRAVEL_COLOR = "#9E9E9E"   # grey   — travel-gap shading

# ── Travel-gap date ranges ────────────────────────────────────────────────────
TRAVEL_PERIODS = [
    (pd.Timestamp("2025-04-01"), pd.Timestamp("2025-04-30"), "Travel 1"),
    (pd.Timestamp("2025-10-08"), pd.Timestamp("2025-10-11"), "Travel 2"),
    (pd.Timestamp("2025-11-22"), pd.Timestamp("2025-12-01"), "Travel 3"),
]

# ── Helper: shade travel gaps on any date-axis axes ───────────────────────────
def shade_travel(ax):
    """Add translucent grey band + italic label for each travel period."""
    ylim = ax.get_ylim()
    for s, e, lbl in TRAVEL_PERIODS:
        ax.axvspan(s, e, color=TRAVEL_COLOR, alpha=0.18)
        ax.text(s + (e - s) / 2, ylim[1] * 0.97, lbl,
                ha="center", fontsize=7, color="#555", style="italic", va="top")

# ── Helper: save figure to figures/ using FIG_FORMAT ─────────────────────────
def savefig(name):
    """Save the current figure to figures/<name>.<FIG_FORMAT> and display inline.

    Parameters
    ----------
    name : str
        Base filename without extension (e.g. ``'fig01_timeseries'``).
        Any extension accidentally included is stripped before appending FIG_FORMAT.
    """
    stem = os.path.splitext(name)[0]          # strip extension if present
    path = os.path.join("figures", f"{stem}.{FIG_FORMAT}")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  ✓ saved → {path}")

print(f"Plotting configuration complete.  FIG_FORMAT = '{FIG_FORMAT}'")


Plotting configuration complete.  FIG_FORMAT = 'svg'


---
## 3 · File Loading & Dataset Preparation

Two input files are loaded:

| File | Description |
|---|---|
| `data/final_combined_power_data.csv` | Daily electricity readings — two columns: `Date`, `Power` (kWh) |
| `data/univ_dates.txt` | One ISO date per line; each line = a university-visit day |

The dataset covers 322 recorded days between 13 March 2025 and 13 March 2026.
Three contiguous gaps correspond to intentional travel absences and are **not imputed**.


In [129]:
# ── 3.1  Load electricity consumption data ────────────────────────────────────
df = pd.read_csv("data/final_combined_power_data.csv")
df.columns = ["Date", "kWh"]
df["Date"]  = pd.to_datetime(df["Date"])
df          = df.sort_values("Date").reset_index(drop=True)

print(f"Rows loaded  : {len(df)}")
print(f"Date range   : {df['Date'].min().date()}  →  {df['Date'].max().date()}")
print(f"kWh range    : {df['kWh'].min():.2f} – {df['kWh'].max():.2f}")
df.head(6)


Rows loaded  : 322
Date range   : 2025-03-13  →  2026-03-13
kWh range    : 0.19 – 3.61


,Date,kWh
0,2025-03-13,0.72
1,2025-03-14,2.38
2,2025-03-15,2.75
3,2025-03-16,2.14
4,2025-03-17,1.88
5,2025-03-18,0.51


In [130]:
# ── 3.2  Load university visit dates ─────────────────────────────────────────
with open("data/univ_dates.txt") as fh:
    lines = [l.strip() for l in fh
             if l.strip() and not l.startswith("#") and l.strip()[0].isdigit()]

univ_dates = set(pd.to_datetime(lines))
print(f"University visit dates loaded: {len(univ_dates)}")


University visit dates loaded: 104


In [131]:
# ── 3.3  Data quality check — verify the three expected travel gaps ───────────
full_range = pd.date_range(df["Date"].min(), df["Date"].max())
missing    = full_range.difference(df["Date"])

print(f"Missing dates within full range: {len(missing)}")
if len(missing):
    gaps = pd.Series(missing).sort_values().reset_index(drop=True)
    blocks = []
    start = end = gaps[0]
    for d in gaps[1:]:
        if (d - end).days == 1:
            end = d
        else:
            blocks.append((start, end, (end - start).days + 1))
            start = end = d
    blocks.append((start, end, (end - start).days + 1))
    print("\nMissing contiguous blocks (travel intervals):")
    for s, e, n in blocks:
        print(f"  {s.date()}  →  {e.date()}  ({n} days)")


Missing dates within full range: 44

Missing contiguous blocks (travel intervals):
  2025-04-01  →  2025-04-30  (30 days)
  2025-10-08  →  2025-10-11  (4 days)
  2025-11-22  →  2025-12-01  (10 days)


---
## 4 · Feature Engineering

Derived columns added to `df`:

| Column | Description |
|---|---|
| `Day_Type` | `"Home"` / `"University"` based on the visit log |
| `Month_Name` | e.g. `"Mar 2025"` — used for axis labels |
| `Day_of_week` | Full weekday name |
| `DOW_num` | 0 = Monday … 6 = Sunday |
| `Week_of_year` | ISO week number |
| `Is_weekend` | `True` for Saturday / Sunday |
| `Year` | Calendar year |

`MONTH_ORDER` is a chronologically sorted list of `Month_Name` values used to  
index all monthly aggregations consistently throughout the notebook.


In [132]:
# ── 4.1  Assign derived columns ───────────────────────────────────────────────
df["Day_Type"]     = df["Date"].apply(
    lambda d: "University" if d in univ_dates else "Home"
)
df["Month_Name"]   = df["Date"].dt.strftime("%b %Y")
df["Day_of_week"]  = df["Date"].dt.day_name()
df["DOW_num"]      = df["Date"].dt.dayofweek           # 0 = Monday
df["Week_of_year"] = df["Date"].dt.isocalendar().week.astype(int)
df["Is_weekend"]   = df["DOW_num"].isin([5, 6])
df["Year"]         = df["Date"].dt.year

# ── Chronological month ordering for all groupby / reindex calls ──────────────
MONTH_ORDER = df.groupby("Month_Name")["Date"].min().sort_values().index.tolist()

# ── Day-type convenience subsets ──────────────────────────────────────────────
home = df[df["Day_Type"] == "Home"]
univ = df[df["Day_Type"] == "University"]

print(f"Total days      : {len(df)}")
print(f"Home Days       : {len(home)}  ({len(home)/len(df)*100:.1f} %)")
print(f"University Days : {len(univ)}  ({len(univ)/len(df)*100:.1f} %)")
df[["Date","kWh","Day_Type","Month_Name","Day_of_week","Is_weekend"]].head(5)


Total days      : 322
Home Days       : 222  (68.9 %)
University Days : 100  (31.1 %)


,Date,kWh,Day_Type,Month_Name,Day_of_week,Is_weekend
0,2025-03-13,0.72,University,Mar 2025,Thursday,False
1,2025-03-14,2.38,Home,Mar 2025,Friday,False
2,2025-03-15,2.75,Home,Mar 2025,Saturday,True
3,2025-03-16,2.14,Home,Mar 2025,Sunday,True
4,2025-03-17,1.88,Home,Mar 2025,Monday,False


---
## 5 · Descriptive Statistics

Standard statistics computed for the full dataset and each day-type subset  
(mean, median, std, min, max, IQR, total sum, count, coefficient of variation).  
Monthly aggregation and the key derived metrics quoted in the paper are also  
produced here so that every downstream figure can reference them as variables.

> **Paper Table 2** is the `stats_df` printout below.  
> **Paper Table 3** is the `monthly` DataFrame.


In [133]:
# ── 5.1  Descriptive statistics helper ───────────────────────────────────────
def describe(series, label):
    q1, q3 = series.quantile([0.25, 0.75])
    return {
        "Group"  : label,
        "N"      : len(series),
        "Mean"   : round(series.mean(),   4),
        "Median" : round(series.median(), 4),
        "Std"    : round(series.std(),    4),
        "Min"    : round(series.min(),    4),
        "Max"    : round(series.max(),    4),
        "IQR"    : round(q3 - q1,         4),
        "Total"  : round(series.sum(),    4),
        "CV (%)" : round(series.std() / series.mean() * 100, 2),
    }

stats_df = pd.DataFrame([
    describe(df["kWh"],   "All Days"),
    describe(home["kWh"], "Home Days"),
    describe(univ["kWh"], "University Days"),
])

print("=" * 70)
print("Table 2 — Descriptive statistics by day type")
print("=" * 70)
print(stats_df.to_string(index=False))
print("=" * 70)


Table 2 — Descriptive statistics by day type
          Group   N   Mean  Median    Std  Min  Max    IQR  Total  CV (%)
       All Days 322 1.5345   1.810 0.7343 0.19 3.61 1.4150 494.10   47.85
      Home Days 222 1.9577   1.955 0.4195 0.20 3.61 0.4275 434.61   21.43
University Days 100 0.5949   0.540 0.2491 0.19 2.31 0.2050  59.49   41.88


In [134]:
# ── 5.2  Key derived metrics quoted in the paper ─────────────────────────────
home_mean = home["kWh"].mean()
univ_mean = univ["kWh"].mean()
diff_kwh  = home_mean - univ_mean
pct_inc   = diff_kwh / univ_mean * 100
avg_watt  = home_mean / 8 * 1000      # assuming 8 active hours/day
max_watt  = home["kWh"].max() / 8 * 1000

print(f"Home mean           : {home_mean:.3f} kWh")
print(f"University mean     : {univ_mean:.3f} kWh")
print(f"Difference          : +{diff_kwh:.3f} kWh  ({pct_inc:.1f}% higher at home)")
print(f"Est. avg wattage    : ~{avg_watt:.0f} W  (8 h active-use assumption)")
print(f"Est. peak wattage   : ~{max_watt:.0f} W  (highest-consumption day)")
print(f"Total annual kWh    : {df['kWh'].sum():.2f} kWh")


Home mean           : 1.958 kWh
University mean     : 0.595 kWh
Difference          : +1.363 kWh  (229.1% higher at home)
Est. avg wattage    : ~245 W  (8 h active-use assumption)
Est. peak wattage   : ~451 W  (highest-consumption day)
Total annual kWh    : 494.10 kWh


In [135]:
# ── 5.3  Monthly aggregation (Table 3 in paper) ───────────────────────────────
monthly = df.groupby("Month_Name").agg(
    Total_kWh = ("kWh",      "sum"),
    Mean_kWh  = ("kWh",      "mean"),
    Days      = ("kWh",      "count"),
    Home_days = ("Day_Type", lambda x: (x == "Home").sum()),
    Univ_days = ("Day_Type", lambda x: (x == "University").sum()),
).reindex(MONTH_ORDER)

print("Table 3 — Monthly energy consumption summary")
monthly


Table 3 — Monthly energy consumption summary


,Total_kWh,Mean_kWh,Days,Home_days,Univ_days
Month_Name,,,,,
Mar 2025,25.63,1.348947,19,12,7
May 2025,45.96,1.482581,31,21,10
Jun 2025,46.34,1.544667,30,19,11
Jul 2025,48.59,1.567419,31,18,13
Aug 2025,47.72,1.539355,31,18,13
Sep 2025,39.00,1.300000,30,16,14
Oct 2025,43.41,1.607778,27,20,7
Nov 2025,33.16,1.579048,21,15,6
Dec 2025,51.49,1.716333,30,25,5


In [ ]:
# ── 5.4  Threshold classifier accuracy (supports Discussion §5.1) ─────────────
# A simple kWh < 0.9 threshold recovers the university-visit log with high accuracy,
# demonstrating that the smart plug signal alone can reconstruct attendance patterns.
THRESHOLD = 0.9
df["Predicted"] = df["kWh"].apply(lambda x: "University" if x < THRESHOLD else "Home")
correct  = (df["Predicted"] == df["Day_Type"]).sum()
accuracy = correct / len(df) * 100

print(f"Threshold classifier  (< {THRESHOLD} kWh → University, else → Home)")
print(f"  Correct predictions : {correct} / {len(df)}")
print(f"  Accuracy            : {accuracy:.1f} %")

from collections import Counter
cm_vals = Counter(zip(df["Day_Type"], df["Predicted"]))
print("\nConfusion matrix:")
for true_lbl in ["Home", "University"]:
    row = {p: cm_vals.get((true_lbl, p), 0) for p in ["Home", "University"]}
    print(f"  True {true_lbl:12s}  →  Home={row['Home']:3d}  University={row['University']:3d}")


---
## 6 · Distribution Analysis

Kernel density estimates are parameterised with `bw_method = 0.25` (Gaussian kernel)
throughout, consistent with the paper's Methodology section.  
The bimodal structure of the combined distribution reflects the two behavioural regimes;
each subgroup individually is approximately unimodal and near-symmetric.

> *Figures 4 and 5 are generated in Section 9.*


---
## 7 · Temporal Analysis

Rolling statistics are computed here; the corresponding figures are in Section 9.

- **7-day centered rolling mean** — short-term trend extraction (Figure 1 overlay)
- **14-day centered rolling mean** — medium-term trend (Figure 1 + Figure 15)
- **14-day rolling standard deviation** — behavioral variability proxy (Figure 15)


In [ ]:
# ── 7.1  Time series with rolling statistics ──────────────────────────────────
ts        = df.set_index("Date")["kWh"]
ma7       = ts.rolling(7,  center=True).mean()
ma14      = ts.rolling(14, center=True).mean()
roll_std  = ts.rolling(14, center=True).std()

print("Rolling windows computed:")
print(f"  7-day  MA  : {ma7.notna().sum()} non-NaN points")
print(f"  14-day MA  : {ma14.notna().sum()} non-NaN points")
print(f"  14-day std : {roll_std.notna().sum()} non-NaN points")


---
## 8 · Clustering Analysis

K-Means with **k = 3** is applied to the univariate daily-kWh series.  
No feature scaling is applied (single variable).  
Clusters are labeled post-hoc by ascending centroid value:
**Idle → Moderate → Heavy**.

> Centroid values, day counts, and percentages reported here are the numbers  
> quoted in the paper's *Behavioral State Clustering* subsection.


In [ ]:
# ── 8.1  K-Means fit ──────────────────────────────────────────────────────────
X  = df[["kWh"]].values
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df["Cluster"] = km.fit_predict(X)

centers   = km.cluster_centers_.flatten()
order     = np.argsort(centers)
label_map = {order[0]: "Idle", order[1]: "Moderate", order[2]: "Heavy"}
df["State"]    = df["Cluster"].map(label_map)
STATE_COLORS   = {"Idle": IDLE_COLOR, "Moderate": MOD_COLOR, "Heavy": HEAVY_COLOR}
sorted_centers = sorted(centers)

print("K-Means cluster centroids (k=3, random_state=42, n_init=10):")
for lbl, c in zip(["Idle", "Moderate", "Heavy"], sorted_centers):
    n = (df["State"] == lbl).sum()
    print(f"  {lbl:10s}: {c:.3f} kWh   ({n} days, {n/len(df)*100:.1f} %)")


---
## 9 · Figure Generation

All paper figures are generated in this section in the same order they appear  
in the published paper. Each subsection header references the paper figure number.  
Output files land in `figures/` with names matching the paper's figure sequence.

| File | Paper Figure |
|---|---|
| `fig01_timeseries` | Figure 1 |
| `fig02_monthly_bars` | Figure 2 |
| `fig03_monthly_trend` | Figure 3 |
| `fig04_histograms` | Figure 4 |
| `fig05_kde_overlap` | Figure 5 |
| `fig06_boxviolin` | Figure 6 |
| `fig07_stripplot` | Figure 7 |
| `fig08_calendar_heatmap` | Figure 8 |
| `fig09_weekly_patterns` | Figure 9 |
| `fig10_heatmap_dow_month` | Figure 10 |
| `fig11_weekday_weekend` | Figure 11 |
| `fig12_kmeans_states` | Figure 12 |
| `fig13_states_by_daytype` | Figure 13 |
| `fig14_cumulative` | Figure 14 |
| `fig15_rolling_variability` | Figure 15 |
| `figA16_day_counts` | Appendix Figure 16 |
| `figA17_dow_scatter` | Appendix Figure 17 |


### Figure 1 — Full-Year Time Series

Color-coded scatter (blue = Home, orange = University) with 7-day (amber) and
14-day (dark green) centered moving averages. Grey shaded bands mark travel gaps.


In [ ]:
fig1, ax = plt.subplots(figsize=(16, 5))

colors = df["Day_Type"].map({"Home": HOME_COLOR, "University": UNIV_COLOR})
ax.scatter(df["Date"], df["kWh"], c=colors, s=25, alpha=0.85, zorder=3)

ax.plot(ma7.index,  ma7.values,  color="#FF6F00", lw=1.8, label="7-day MA",  zorder=4)
ax.plot(ma14.index, ma14.values, color="#1B5E20", lw=2.0, label="14-day MA", zorder=4)

shade_travel(ax)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha="right")

h_p = mpatches.Patch(color=HOME_COLOR,  label="Home Day")
u_p = mpatches.Patch(color=UNIV_COLOR,  label="University Day")
t_p = mpatches.Patch(color=TRAVEL_COLOR, alpha=0.4, label="Travel Period")
ax.legend(handles=[h_p, u_p, t_p,
                   plt.Line2D([], [], color="#FF6F00", lw=2, label="7-day MA"),
                   plt.Line2D([], [], color="#1B5E20", lw=2, label="14-day MA")],
          loc="upper left", fontsize=9, ncol=3)

ax.set_title("Full-Year PC Electricity Consumption Time Series", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Consumption (kWh)")
plt.tight_layout()
savefig("fig01_timeseries")


### Figure 2 — Monthly Energy Breakdown

Left: total kWh per calendar month.  
Right: monthly totals split by Home (blue) vs University (orange) days.


In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(16, 5))
colors_m   = plt.cm.viridis(np.linspace(0.15, 0.85, len(monthly)))

# Left panel — total kWh per month
ax = axes[0]
bars = ax.bar(range(len(monthly)), monthly["Total_kWh"], color=colors_m, edgecolor="white")
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly.index, rotation=35, ha="right", fontsize=9)
ax.set_title("Total PC Energy per Month", fontweight="bold")
ax.set_ylabel("Total kWh")
for bar, val in zip(bars, monthly["Total_kWh"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}", ha="center", va="bottom", fontsize=8)

# Right panel — Home vs University energy split
ax    = axes[1]
x     = np.arange(len(monthly))
w     = 0.42
home_kwh = df[df["Day_Type"]=="Home"].groupby("Month_Name")["kWh"].sum().reindex(MONTH_ORDER).fillna(0)
univ_kwh = df[df["Day_Type"]=="University"].groupby("Month_Name")["kWh"].sum().reindex(MONTH_ORDER).fillna(0)
ax.bar(x - w/2, home_kwh.values, w, label="Home",       color=HOME_COLOR, alpha=0.88, edgecolor="white")
ax.bar(x + w/2, univ_kwh.values, w, label="University", color=UNIV_COLOR, alpha=0.88, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(monthly.index, rotation=35, ha="right", fontsize=9)
ax.set_title("Home vs University Energy per Month", fontweight="bold")
ax.set_ylabel("Total kWh")
ax.legend()

plt.suptitle("Monthly Energy Breakdown", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
savefig("fig02_monthly_bars")


### Figure 3 — Monthly Average Daily Consumption Trend

Three traces: overall mean (purple), Home Day mean (blue dashed), University Day mean (orange dashed).
The shaded band between Home and University averages highlights the persistent gap across all months.


In [ ]:
fig3, ax = plt.subplots(figsize=(14, 5))
home_avg = df[df["Day_Type"]=="Home"].groupby("Month_Name")["kWh"].mean().reindex(MONTH_ORDER)
univ_avg = df[df["Day_Type"]=="University"].groupby("Month_Name")["kWh"].mean().reindex(MONTH_ORDER)

ax.plot(range(len(monthly)), monthly["Mean_kWh"],   "o-",  color="#673AB7", lw=2.5, ms=8,  label="Overall Avg", zorder=3)
ax.plot(range(len(monthly)), home_avg.values, "s--", color=HOME_COLOR,      lw=1.8, ms=7,  label="Home Avg",    alpha=0.85)
ax.plot(range(len(monthly)), univ_avg.values, "D--", color=UNIV_COLOR,      lw=1.8, ms=7,  label="Univ Avg",    alpha=0.85)
ax.fill_between(range(len(monthly)), home_avg.values, univ_avg.values, alpha=0.12, color="#673AB7")
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly.index, rotation=35, ha="right", fontsize=9)
ax.set_title("Monthly Average Daily PC Consumption Trend", fontweight="bold", fontsize=13)
ax.set_ylabel("Mean kWh / day")
ax.legend(fontsize=10)
plt.tight_layout()
savefig("fig03_monthly_trend")


### Figure 4 — Distribution Histograms with KDE

Three panels: All Days (bimodal combined), Home Days (unimodal ~1.5–2.5 kWh),  
University Days (unimodal ~0.2–0.8 kWh). KDE bandwidth = 0.25 throughout.


In [ ]:
fig4, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, subset, color, label in [
    (axes[0], df,   "#7E57C2", "All Days"),
    (axes[1], home, HOME_COLOR, "Home Days"),
    (axes[2], univ, UNIV_COLOR, "University Days"),
]:
    data = subset["kWh"].values
    ax.hist(data, bins=25, color=color, alpha=0.6, edgecolor="white", density=True)
    kde = gaussian_kde(data, bw_method=0.25)
    xs  = np.linspace(data.min() - 0.1, data.max() + 0.1, 300)
    ax.plot(xs, kde(xs), color=color, lw=2.5)
    ax.axvline(data.mean(),       color="black", lw=1.5, ls="--", label=f"Mean={data.mean():.2f}")
    ax.axvline(np.median(data),   color="grey",  lw=1.5, ls=":",  label=f"Median={np.median(data):.2f}")
    ax.set_title(f"{label} Distribution", fontweight="bold")
    ax.set_xlabel("kWh")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.suptitle("Distribution of Daily PC Energy Consumption", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("fig04_histograms")


### Figure 5 — Overlapping KDE Distributions (Home vs University)

Near-complete separation between the two distributions is visible:
the University Day mode (~0.55 kWh) and the Home Day mode (~1.80–2.00 kWh) are disjoint.


In [ ]:
fig5, ax = plt.subplots(figsize=(10, 5))
for subset, color, label in [(home, HOME_COLOR, "Home"), (univ, UNIV_COLOR, "University")]:
    data = subset["kWh"].values
    ax.hist(data, bins=22, color=color, alpha=0.3, density=True, edgecolor="white")
    kde = gaussian_kde(data, bw_method=0.25)
    xs  = np.linspace(0, 3.5, 400)
    ax.fill_between(xs, kde(xs), alpha=0.18, color=color)
    ax.plot(xs, kde(xs), color=color, lw=2.5, label=label)

ax.set_xlabel("Daily Consumption (kWh)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Home vs University — Overlapping KDE Distributions", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
savefig("fig05_kde_overlap")


### Figure 6 — Box Plot and Violin Plot (Home vs University)

Notched box plot (left) and violin plot (right) confirm the large location shift  
and the difference in spread between the two day-type distributions.


In [ ]:
fig6, axes = plt.subplots(1, 2, figsize=(13, 5))

# Notched box plot
ax    = axes[0]
bplot = ax.boxplot(
    [home["kWh"].values, univ["kWh"].values],
    patch_artist=True, notch=True,
    medianprops=dict(color="yellow", linewidth=2.5)
)
for patch, c in zip(bplot["boxes"], [HOME_COLOR, UNIV_COLOR]):
    patch.set_facecolor(c); patch.set_alpha(0.75)
ax.set_xticklabels(["Home", "University"], fontsize=12)
ax.set_ylabel("kWh")
ax.set_title("Box Plot: Home vs University", fontweight="bold")

# Violin plot
ax = axes[1]
vp = ax.violinplot([home["kWh"].values, univ["kWh"].values],
                   positions=[1, 2], showmeans=True, showmedians=True, widths=0.6)
for pc, c in zip(vp["bodies"], [HOME_COLOR, UNIV_COLOR]):
    pc.set_facecolor(c); pc.set_alpha(0.7)
vp["cmeans"].set_color("gold"); vp["cmedians"].set_color("white")
ax.set_xticks([1, 2]); ax.set_xticklabels(["Home", "University"], fontsize=12)
ax.set_ylabel("kWh")
ax.set_title("Violin Plot: Home vs University", fontweight="bold")

plt.suptitle("Home vs University Day Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("fig06_boxviolin")


### Figure 7 — Strip Plot with Mean Lines

Jittered scatter of individual daily observations. Horizontal black lines mark the group mean.
The visual gap between clusters illustrates the near-zero overlap between day types.


In [ ]:
fig7, ax = plt.subplots(figsize=(9, 5))
np.random.seed(42)
for i, (subset, color, label) in enumerate(
        [(home, HOME_COLOR, "Home"), (univ, UNIV_COLOR, "University")]):
    jitter = np.random.uniform(-0.15, 0.15, len(subset))
    ax.scatter(jitter + i, subset["kWh"], color=color, alpha=0.5, s=30, label=label)
    ax.hlines(subset["kWh"].mean(), i - 0.25, i + 0.25, color="black", lw=2.5, zorder=5)

ax.set_xticks([0, 1]); ax.set_xticklabels(["Home", "University"], fontsize=12)
ax.set_ylabel("kWh")
ax.set_title("Strip Plot with Mean Lines", fontweight="bold", fontsize=13)
ax.legend()
plt.tight_layout()
savefig("fig07_stripplot")


### Figure 8 — Calendar Heatmap

Month-by-month grid (Mon–Sun columns). Each cell shows the daily kWh reading on a  
Red–Yellow–Green diverging scale. ✈ marks travel-gap days within the dataset range;  
cells outside the observation window are blank.


In [ ]:
df_cal     = df.set_index("Date")["kWh"]
data_start = df["Date"].min()
data_end   = df["Date"].max()
months_plot = pd.date_range("2025-03-01", "2026-03-31", freq="MS")
ncols = 4
nrows = (len(months_plot) + ncols - 1) // ncols
vmin, vmax = df["kWh"].min(), df["kWh"].max()
cmap = plt.cm.RdYlGn

fig8, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 2.8))
axes = axes.flatten()

for idx, month_start in enumerate(months_plot):
    ax        = axes[idx]
    month_end = month_start + pd.offsets.MonthEnd(0)
    days_in_month = pd.date_range(month_start, month_end)
    first_dow  = month_start.dayofweek
    total_cells = first_dow + len(days_in_month)
    n_weeks     = int(np.ceil(total_cells / 7))
    grid        = np.full((n_weeks, 7), np.nan)

    for i, day in enumerate(days_in_month):
        r = (first_dow + i) // 7
        c = (first_dow + i) % 7
        if day in df_cal.index:
            grid[r, c] = df_cal[day]

    masked = np.ma.masked_invalid(grid)
    ax.imshow(masked, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")

    for r in range(n_weeks):
        for c in range(7):
            di = r * 7 + c - first_dow
            if 0 <= di < len(days_in_month):
                day = days_in_month[di]
                val = grid[r, c]
                if not np.isnan(val):
                    ax.text(c, r, f"{val:.1f}", ha="center", va="center",
                            fontsize=6, color="black" if val > 1.0 else "white")
                elif data_start <= day <= data_end:
                    ax.text(c, r, "✈", ha="center", va="center",
                            fontsize=16, color="#444", alpha=0.9)

    ax.set_xticks(range(7))
    ax.set_xticklabels(["M","T","W","T","F","S","S"], fontsize=8)
    ax.set_yticks([])
    ax.set_title(month_start.strftime("%b %Y"), fontsize=9, fontweight="bold")

for idx in range(len(months_plot), len(axes)):
    axes[idx].axis("off")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax))
sm.set_array([])
fig8.colorbar(sm, ax=axes[:len(months_plot)], orientation="horizontal",
              fraction=0.015, pad=0.08, label="kWh")
fig8.suptitle("Calendar Heatmap — Daily PC Energy Consumption (kWh)",
              fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
savefig("fig08_calendar_heatmap")


### Figure 9 — Weekly Patterns (Day-of-Week)

Left: box plots per weekday showing spread and median.  
Right: mean consumption per weekday. Weekend columns (Sat/Sun) are highlighted.  
University visits on weekdays suppress Tuesday–Thursday averages most strongly.


In [ ]:
DOW_ORDER  = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
colors_dow = plt.cm.Set3(np.linspace(0, 1, 7))

fig9, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: box plots per weekday
ax       = axes[0]
dow_data = [df[df["Day_of_week"] == d]["kWh"].values for d in DOW_ORDER]
bplot    = ax.boxplot(dow_data, patch_artist=True,
                      medianprops=dict(color="black", linewidth=2))
for patch, c in zip(bplot["boxes"], colors_dow):
    patch.set_facecolor(c); patch.set_alpha(0.85)
ax.set_xticklabels([d[:3] for d in DOW_ORDER], fontsize=10)
ax.set_ylabel("kWh")
ax.set_title("Weekly Pattern: Box Plots", fontweight="bold")
ax.axvspan(5.5, 7.5, color="#FFECB3", alpha=0.4, label="Weekend")
ax.legend()

# Right: mean per weekday
ax        = axes[1]
dow_means = [df[df["Day_of_week"] == d]["kWh"].mean() for d in DOW_ORDER]
bars      = ax.bar(DOW_ORDER, dow_means, color=colors_dow, edgecolor="white")
ax.set_xticklabels([d[:3] for d in DOW_ORDER], fontsize=10)
ax.set_ylabel("Mean kWh")
ax.set_title("Average Consumption by Weekday", fontweight="bold")
for bar, val in zip(bars, dow_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9)
ax.axvspan(4.5, 6.5, color="#FFECB3", alpha=0.4)

plt.suptitle("Day-of-Week Energy Patterns", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("fig09_weekly_patterns")


### Figure 10 — Heatmap: Mean kWh by Day-of-Week × Month

Rows = weekday, columns = calendar month. Lighter cells = lower mean consumption.
University-heavy months (Sep 2025) show systematically lower weekday readings.


In [ ]:
fig10, ax = plt.subplots(figsize=(14, 5))
pivot = (df.pivot_table(values="kWh", index="Day_of_week",
                        columns="Month_Name", aggfunc="mean")
           .reindex(DOW_ORDER)[MONTH_ORDER])
sns.heatmap(pivot, cmap="YlOrRd", annot=True, fmt=".2f",
            linewidths=0.5, linecolor="white", ax=ax,
            cbar_kws={"label": "Mean kWh"})
ax.set_title("Heatmap: Mean kWh by Day-of-Week × Month", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Day of Week")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
savefig("fig10_heatmap_dow_month")


### Figure 11 — Weekday vs Weekend Comparison

Violin plot (left), overlapping KDE (centre), and mean ± std bar chart (right).
Weekends show slightly higher mean consumption than weekdays, attributable to the  
absence of university visits on weekends suppressing the weekday average.


In [ ]:
weekend = df[df["Is_weekend"]]
weekday = df[~df["Is_weekend"]]

fig11, axes = plt.subplots(1, 3, figsize=(15, 5))

# Violin
ax = axes[0]
vp = ax.violinplot([weekday["kWh"].values, weekend["kWh"].values],
                   positions=[1, 2], showmeans=True, showmedians=True, widths=0.5)
for pc, c in zip(vp["bodies"], ["#1976D2", "#E64A19"]):
    pc.set_facecolor(c); pc.set_alpha(0.7)
ax.set_xticks([1, 2]); ax.set_xticklabels(["Weekday", "Weekend"])
ax.set_ylabel("kWh")
ax.set_title("Violin: Weekday vs Weekend", fontweight="bold")

# KDE
ax = axes[1]
for subset, color, label in [(weekday, "#1976D2", "Weekday"), (weekend, "#E64A19", "Weekend")]:
    data = subset["kWh"].values
    kde  = gaussian_kde(data, bw_method=0.25)
    xs   = np.linspace(0, 3.5, 300)
    ax.fill_between(xs, kde(xs), alpha=0.22, color=color)
    ax.plot(xs, kde(xs), color=color, lw=2.5, label=label)
ax.set_xlabel("kWh"); ax.set_ylabel("Density")
ax.set_title("KDE: Weekday vs Weekend", fontweight="bold")
ax.legend()

# Mean ± std bar chart
ax    = axes[2]
means = [weekday["kWh"].mean(), weekend["kWh"].mean()]
stds  = [weekday["kWh"].std(),  weekend["kWh"].std()]
bars  = ax.bar(["Weekday", "Weekend"], means,
               color=["#1976D2", "#E64A19"],
               yerr=stds, capsize=6, edgecolor="white", error_kw={"lw": 2})
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", va="bottom", fontsize=11)
ax.set_ylabel("Mean kWh")
ax.set_title("Mean ± Std: Weekday vs Weekend", fontweight="bold")

plt.suptitle("Weekday vs Weekend Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("fig11_weekday_weekend")


### Figure 12 — K-Means Behavioral States (Time Series)

Left: full-year scatter coloured by state (Idle = purple, Moderate = green, Heavy = red).  
Dashed horizontal lines mark the cluster centroids.  
Right: day-count bar chart per state.


In [ ]:
fig12, axes = plt.subplots(1, 2, figsize=(15, 5))

# Time-series coloured by state
ax = axes[0]
for state, color in STATE_COLORS.items():
    sub = df[df["State"] == state]
    ax.scatter(sub["Date"], sub["kWh"], c=color, s=28, alpha=0.8, label=state, zorder=3)
shade_travel(ax)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
for lbl, c in zip(["Idle","Moderate","Heavy"], sorted_centers):
    ax.axhline(c, ls="--", lw=1, color=STATE_COLORS[lbl], alpha=0.5)
ax.legend(); ax.set_ylabel("kWh")
ax.set_title("K-Means Behavioral States (Time Series)", fontweight="bold")

# Count bar chart
ax          = axes[1]
state_counts = df["State"].value_counts()[["Idle","Moderate","Heavy"]]
bars         = ax.bar(state_counts.index, state_counts.values,
                      color=[STATE_COLORS[s] for s in state_counts.index],
                      edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, state_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of Days")
ax.set_title("Count by Behavioral State", fontweight="bold")

plt.suptitle("Behavioral State Clustering via K-Means", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("fig12_kmeans_states")


### Figure 13 — Behavioral States by Day Type

Left: stacked bar — how Home and University days distribute across Idle / Moderate / Heavy states.  
Right: monthly stacked bar showing the time-evolution of state composition.  
The Idle cluster is nearly co-extensive with the University Day set.


In [ ]:
fig13, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar: Home vs University per state
ax = axes[0]
ct = pd.crosstab(df["Day_Type"], df["State"])[["Idle","Moderate","Heavy"]]
ct.plot(kind="bar", stacked=True,
        color=[IDLE_COLOR, MOD_COLOR, HEAVY_COLOR],
        ax=ax, edgecolor="white", rot=0)
ax.set_title("Behavioral States: Home vs University", fontweight="bold")
ax.set_ylabel("Days"); ax.legend(title="State")

# Monthly state distribution
ax            = axes[1]
monthly_state = (df.groupby("Month_Name")["State"]
                   .value_counts().unstack()
                   .reindex(MONTH_ORDER).fillna(0))
monthly_state[["Idle","Moderate","Heavy"]].plot(
    kind="bar", stacked=True,
    color=[IDLE_COLOR, MOD_COLOR, HEAVY_COLOR],
    ax=ax, edgecolor="white", rot=35, width=0.7)
ax.set_title("Behavioral States by Month", fontweight="bold")
ax.set_ylabel("Days"); ax.legend(title="State")
ax.tick_params(axis="x", labelsize=8)
ax.get_legend().remove()

plt.suptitle("Behavioral State Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("fig13_states_by_daytype")


### Figure 14 — Cumulative Energy Consumption

The curve rises at ~1.53 kWh/day on average, reaching 494.1 kWh by 13 March 2026.  
Three flat plateaus correspond to the travel gaps; the April 2025 plateau is the most prominent  
(~60 kWh of consumption that did not occur during the Japan research visit).


In [ ]:
df_sorted            = df.sort_values("Date").copy()
df_sorted["Cumulative"] = df_sorted["kWh"].cumsum()

fig14, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_sorted["Date"], df_sorted["Cumulative"], color="#673AB7", lw=2.5,
        label="Cumulative kWh", zorder=3)
ax.fill_between(df_sorted["Date"], df_sorted["Cumulative"], alpha=0.1, color="#673AB7")
shade_travel(ax)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha="right")
ax.set_title("Cumulative PC Energy Consumption Over the Year", fontsize=13, fontweight="bold")
ax.set_ylabel("Cumulative kWh"); ax.set_xlabel("Date"); ax.legend()
plt.tight_layout()
savefig("fig14_cumulative")

print(f"Total annual consumption: {df['kWh'].sum():.2f} kWh")


### Figure 15 — 14-Day Rolling Mean & Behavioral Variability

Top panel: raw scatter (colored by day type) with 14-day rolling mean ± 1 std band.  
Bottom panel: 14-day rolling std as a standalone variability trace.  
Highest variability occurs in the October–November travel transition period.


In [ ]:
c_scatter = df["Day_Type"].map({"Home": HOME_COLOR, "University": UNIV_COLOR})

fig15, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

ax = axes[0]
ax.scatter(df["Date"], df["kWh"], c=c_scatter.values, s=20, alpha=0.6, zorder=2)
ax.plot(ma14.index, ma14.values, color="black", lw=2, label="14-day mean", zorder=3)
ax.fill_between(ma14.index,
                ma14.values - roll_std.values,
                ma14.values + roll_std.values,
                alpha=0.18, color="black", label="±1 std band")
shade_travel(ax)
ax.set_ylabel("kWh")
ax.set_title("14-day Rolling Mean ± Std Band", fontweight="bold")
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(roll_std.index, roll_std.values, color="#E91E63", lw=2)
ax.fill_between(roll_std.index, 0, roll_std.values, alpha=0.2, color="#E91E63")
shade_travel(ax)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
ax.set_ylabel("Rolling Std")
ax.set_title("14-day Rolling Std — Behavioral Variability Over Time", fontweight="bold")

plt.suptitle("Temporal Variability Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("fig15_rolling_variability")


---
## Appendix · Additional Figures

Supplementary visualizations referenced in the paper's appendix section.


### Appendix Figure 16 — Home vs University Day Counts per Month


In [ ]:
fig16, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(monthly))
w = 0.4
ax.bar(x - w/2, monthly["Home_days"], w, label="Home Days",       color=HOME_COLOR, alpha=0.85, edgecolor="white")
ax.bar(x + w/2, monthly["Univ_days"], w, label="University Days", color=UNIV_COLOR, alpha=0.85, edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(monthly.index, rotation=35, ha="right", fontsize=9)
ax.set_title("Home vs University Day Counts per Month", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Days"); ax.legend()
plt.tight_layout()
savefig("figA16_day_counts")


### Appendix Figure 17 — Scatter: kWh vs Day-of-Week (Home / University)


In [ ]:
fig17, ax = plt.subplots(figsize=(10, 5))
c_scatter2 = df["Day_Type"].map({"Home": HOME_COLOR, "University": UNIV_COLOR})
ax.scatter(df["DOW_num"], df["kWh"], c=c_scatter2, s=40, alpha=0.5)
ax.set_xticks(range(7))
ax.set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax.set_ylabel("kWh"); ax.set_xlabel("Day of Week")
ax.set_title("kWh vs Day of Week (Home / University)", fontweight="bold", fontsize=13)
ax.legend(handles=[mpatches.Patch(color=HOME_COLOR, label="Home"),
                   mpatches.Patch(color=UNIV_COLOR, label="University")])
plt.tight_layout()
savefig("figA17_dow_scatter")


---
## Figure Inventory

Verify all expected output files are present after a full run.


In [ ]:
import glob
figs = sorted(glob.glob(f"figures/*.{FIG_FORMAT}"))
print(f"Total figures saved to figures/  [{FIG_FORMAT.upper()}]: {len(figs)}")
for f in figs:
    size_kb = os.path.getsize(f) // 1024
    print(f"  {os.path.basename(f):45s}  {size_kb:4d} KB")
full_range = pd.date_range("2025-03-13", "2026-03-13")
missing = full_range.difference(df["Date"])
print(f"Total missing days: {len(missing)}")
print("\nMissing dates:")
for d in missing:
    print(d.date())
print(len(df))